<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 04 · Feature Thinking

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
Chapter 6 treats features as design choices. The same rows can look very
different to a model depending on how the inputs are represented.

This cell builds the supervised-learning baseline by selecting features, keeping the target separate, and preparing a train/test split.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "3_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

module_path = BOOK_ROOT / 'code' / 'churn_report.py'
spec = importlib.util.spec_from_file_location('churn_report', module_path)
assert spec is not None and spec.loader is not None
churn_report = importlib.util.module_from_spec(spec)
spec.loader.exec_module(churn_report)

print(f'Loaded {module_path.resolve()}')


This cell groups the data and orders the result so the learner can compare categories clearly.


In [ ]:
df = churn_report.load_customers()

print(df.head().to_string(index=False))
print()
print(df.dtypes.to_string())
print()
print('churn rate by plan type:')
print(df.groupby('plan_type')['churned'].mean().sort_values(ascending=False))
print()
print('churn rate by country:')
print(df.groupby('country')['churned'].mean().sort_values(ascending=False))


This cell builds the supervised-learning baseline by selecting features, keeping the target separate, and preparing a train/test split.


In [ ]:
feature_frame = df[churn_report.FEATURE_COLUMNS]
target = df[churn_report.TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(  # split off held-out data
    feature_frame,
    target,
    test_size=0.25,
    random_state=42,
    stratify=target,
)

preprocessor = ColumnTransformer(  # scale numeric features and one-hot encode categoricals
    [
        ('num', StandardScaler(), churn_report.NUMERIC_COLUMNS),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), churn_report.CATEGORICAL_COLUMNS),
    ]
)
pipe = Pipeline(
    [
        ('pre', preprocessor),
        ('model', LogisticRegression(max_iter=1000)),
    ]
)
pipe.fit(X_train, y_train)

feature_names = pipe.named_steps['pre'].get_feature_names_out()
coef_frame = pd.Series(pipe.named_steps['model'].coef_.ravel(), index=feature_names)
coef_frame = coef_frame.sort_values(key=lambda s: s.abs(), ascending=False)

print('transformed shape =', pipe.named_steps['pre'].transform(X_train).shape)
print()
print(coef_frame.head(12).to_string())


This cell continues the worked example for the current section.


In [ ]:
transformed_row = pipe.named_steps['pre'].transform(X_test.iloc[[0]])
print(transformed_row)


### Where We Are Heading Next
Chapter 7 steps back from individual features and introduces clustering
and dimensionality reduction as exploration tools.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
